### E-Commerce US dataset

\Importing Necessary Libraries

In [21]:
import numpy as np
import pandas as pd
import os

\Loading the Sampled dataset

In [22]:
SAMPLE_PATH = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\raw_sample"
tables_name=["customers","orders","order_items","order_payments","order_reviews","products","sellers","geolocation"]

tables={}
for name in tables_name:
    tables[name]=pd.read_csv(os.path.join(SAMPLE_PATH,f"{name}.csv"))

customers=tables["customers"]
orders=tables["orders"]
order_items=tables["order_items"]
order_payments=tables["order_payments"]
order_reviews=tables["order_reviews"]
products=tables["products"]
sellers=tables["sellers"]
geolocation=tables["geolocation"]


\checking data explosion before merging the files

In [23]:
items_enriched = (order_items
    .merge(products,on="product_id",how="left",suffixes=("","_prod"))
    .merge(sellers,on="seller_id",how="left",suffixes=("","_sell"))
                 )

print("order_items :",order_items.shape[0],"rows")
print("items_enriched :", items_enriched.shape[0],"rows")
print("There is no data explosion" if order_items.shape[0]==items_enriched.shape[0] else "There is data explosion" )

order_items : 21838 rows
items_enriched : 21838 rows
There is no data explosion


**Insights:** 

Merging order_items with products (on product_id) and then with sellers (on seller_id) returns exactly 21,838 rows, matching the source order_items count.

No Row Duplication(fan_out) occurred, confirming that product_id and seller_id are clean M:1 foreign keys from the order_items perspective. 

items_enriched is safe to use as the enriched line-item table without any row-count deduplication step.

In [24]:
items_enriched.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,discount_rate,product_category_name,product_name,...,product_width_cm,cost,price_prod,seller_company_name,seller_contact_name,seller_contact_gender,seller_contact_age,seller_zip_code_prefix,seller_city,seller_state
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1,89804d82-33a1-4558-8fa2-9b0252a2a406,36c2b043-ccf4-4d9c-a0bb-4b8d28737fd7,2025-12-30 07:07:20,916.90,45.47,0.0,furniture,Nova Desk - Industrial,...,13,395.24,916.90,Jones-Russell,Nicholas Thompson,M,18,90086,Los Angeles,CA
1,720fd9e9-cdeb-4dec-a187-f71586eb085a,2,ac10b78c-342e-4b50-9ea9-bddde7db2e79,09936504-415b-4f90-a5b8-ad15a5be67d0,2025-12-30 07:07:20,817.14,172.48,0.0,electronics,Vertex Smartwatch S468,...,23,717.81,817.14,Crawford-Pugh,Jeffrey Stone,M,70,19137,Philadelphia,PA
2,720fd9e9-cdeb-4dec-a187-f71586eb085a,3,f18e6310-0e75-4b7f-bc20-90ac1e7bd466,7d336012-7101-4a66-b6bc-c269620d8df9,2025-12-30 07:07:20,37.04,94.98,0.0,fashion,Crest Minimal Dress,...,7,9.68,37.04,Fields PLC,Jill Williams,F,27,98129,Seattle,WA
3,c0142972-63fa-4af2-8070-f583ab769847,1,5816f107-b725-4c41-b794-298bf9669a41,91fe2cc8-51a5-4c79-9c12-cea5ae013f55,2019-06-10 19:30:44,917.98,27.42,0.0,furniture,Zenith Oak Coffee Table,...,23,587.82,917.98,Johnson-Garcia,Michelle Gentry,F,19,90021,Los Angeles,CA
4,c0142972-63fa-4af2-8070-f583ab769847,2,f4b3ce27-568d-4c33-9248-6c314635f80a,54165da9-b969-4d50-9b61-ffd0d8b4d731,2019-06-10 19:30:44,28.71,145.25,0.0,toys,Nimbus Doll Toy,...,10,15.45,28.71,Alvarez-Phillips,Tyler Johnson,M,34,60660,Chicago,IL


**Insights:** 

Each row in items_enriched represents one ordered line item enriched with product attributes (category, brand, dimensions, cost) and seller details (company, city, state). 

The discount_rate is 0.0 across all visible rows, indicating no discounts applied in this slice of the sample. 

price and price_prod are identical, as expected since price_prod captures the product catalog price at purchase time.

Seller locations span CA, PA, WA, and IL, showing multi-state seller distribution within just the first five rows.

\ one to many tables -- doing groupby and aggregation to bring values into same row

In [25]:
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        n_items = ("order_id","count"),
        total_price = ("price","sum"),
        total_freight = ("freight_value","sum"),
        avg_price = ("price","mean"),
        n_unique_products = ("product_id","nunique"),
        n_unique_sellers = ("seller_id", "nunique"),
).reset_index()
)
items_agg.shape

(10000, 7)

In [26]:
pay_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment = ("payment_value","sum"),
        max_installments = ("payment_installments", "max"),
        n_payment_types = ("payment_type","nunique"),
        main_payment_type = ("payment_type", lambda x:x.mode().iloc[0]),
    ).reset_index()
)
pay_agg.shape

(10000, 5)

In [27]:
rev_agg = (
    order_reviews
    .groupby("order_id")
    .agg(
        avg_review_score = ("review_score", "mean"),
        n_reviews = ("review_id", "count"),
    ).reset_index()
)
rev_agg.shape

(9332, 3)

In [28]:
geo_agg = (
    geolocation
    .groupby("zip_code_prefix")
    .agg(
        geo_lat = ("geolocation_lat","mean"),
        geo_lng = ("geolocation_lng","mean"),
        geo_city = ("geolocation_city","first"),
        geo_state = ("geolocation_state", "first"),
    ).reset_index()
)
geo_agg.shape

(849, 5)

In [29]:
items_cat = order_items.merge(products[["product_id","product_category_name","product_brand"]],
                              on="product_id", how ="left"
                             )
cat_agg= (
    items_cat
    .groupby("order_id")
    .agg(
        main_category = ("product_category_name", lambda x: x.mode().loc[0] if len(x.mode())>0 else None),
        main_brand = ("product_brand", lambda x: x.mode().loc[0] if len(x.mode())>0 else None),
        n_category = ("product_category_name", "nunique"),
    ).reset_index()
)
cat_agg.shape

(10000, 4)

**Insights:**
 
 All five groupby aggregations (items, payments, reviews, geolocation, category) completed cleanly. 
 
 items_agg, pay_agg, and cat_agg each return exactly 10,000 rows confirming full order coverage for financial and category metrics. 
 
 rev_agg is the only exception at 9,332 rows, meaning 668 orders have no review record and will appear as nulls after the master merge. 
 
 geo_agg resolved 849 unique zip codes to average lat/lng coordinates for spatial joins.

\Intergrating the data

In [30]:
master = (
    orders
    .merge(customers,on= "customer_id",how="left")
    .merge(items_agg,on="order_id",how="left")
    .merge(pay_agg,on="order_id",how="left")
    .merge(rev_agg,on="order_id",how="left")
    .merge(geo_agg,left_on="customer_zip_code_prefix",right_on="zip_code_prefix",how="left")
    .merge(cat_agg,on="order_id",how="left")
)

In [31]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_name,...,avg_review_score,n_reviews,zip_code_prefix,geo_lat,geo_lng,geo_city,geo_state,main_category,main_brand,n_category
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaN,NaN,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,...,NaN,NaN,85037,33.440476,-112.048724,Phoenix,AZ,electronics,Crest,3
1,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,...,5.0,1.0,90060,34.064636,-118.225773,Los Angeles,CA,furniture,Nimbus,3
2,11bdf634-2b87-4d37-8d76-be1e7aff8f3b,89b0d980-868f-478d-a63b-5fea5f265f4f,delivered,2023-04-02 14:39:33,2023-04-02 23:42:33,2023-04-03 23:42:33,2023-04-08 23:42:33,2023-04-10 23:42:33,c820d3a4-edd4-4b32-9605-22d97dcc8e5e,Debra Jones,...,5.0,1.0,30398,33.828795,-84.402829,Atlanta,GA,electronics,Crest,3
3,d29b58ad-af2a-47b4-9214-b2664fb1fdba,336d14e6-a22f-4126-b2b7-fa64a2955b34,delivered,2023-01-28 12:09:24,2023-01-28 15:26:24,2023-01-29 15:26:24,2023-02-02 15:26:24,2023-02-04 15:26:24,d3d86bff-2f03-4bec-9cbf-9a2f8f0d5783,Jamie Estrada,...,5.0,1.0,30352,33.787245,-84.387726,Atlanta,GA,books,Crest,2
4,53fabbc8-fd4f-4c12-8af9-258db94dda6c,340c9f9d-bb72-4bfa-a6aa-aa9c7c18a03f,canceled,2019-06-05 22:33:47,2019-06-06 01:24:47,NaN,NaN,2019-06-16 01:24:47,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,...,NaN,NaN,90060,34.064636,-118.225773,Los Angeles,CA,books,Crest,3


In [32]:
print(f"master after geo merge: {master.shape[0]} rows x {master.shape[1]} cols")
print(f"No explode? {master.shape[0] == 10000} ")
print(f"Customers with map coordinates: {master['geo_lat'].notna().sum()}/{len(master)}")

master after geo merge: 10000 rows x 36 cols
No explode? True 
Customers with map coordinates: 10000/10000


**Insights:** 

The final master table holds 10,000 rows across 36 columns with zero row inflation from any of the six LEFT joins.

Every order has a valid geo_lat value, meaning all 10,000 customer zip codes matched at least one entry in the geolocation table. 

The 36 columns span order lifecycle, customer profile, aggregated financials, payment method, review scores, and product category in a single flat table. 

This makes master directly usable for analysis without further joins.

\Saving the integrated data

In [33]:
out_path=r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\raw_integrated"

master.to_csv(os.path.join(out_path,"master_orders.csv"),index =False)
items_enriched.to_csv(os.path.join(out_path,"items_enriched.csv"),index =False)

\Relationship Mapping Document

In [34]:
relationship_map = pd.DataFrame([
    {"table_left": "orders","join_key": "customer_id","table_right": "customers","join_type": "LEFT","cardinality": "M:1","notes": "Each order belongs to one customer"},
    {"table_left": "orders","join_key": "order_id","table_right": "items_agg","join_type": "LEFT","cardinality": "1:1","notes": "Aggregated order item metrics per order"},
    {"table_left": "orders","join_key": "order_id","table_right": "pay_agg","join_type": "LEFT","cardinality": "1:1","notes": "Aggregated payment metrics per order"},
    {"table_left": "orders","join_key": "order_id","table_right": "rev_agg","join_type": "LEFT","cardinality": "1:1","notes": "Aggregated review metrics per order"},
    {"table_left": "customers","join_key": "customer_zip_code_prefix","table_right": "geo_agg","join_type": "LEFT","cardinality": "M:1","notes": "Customer location attributes aggregated by zip code"},
    {"table_left": "orders","join_key": "order_id","table_right": "cat_agg","join_type": "LEFT","cardinality": "1:1","notes": "Aggregated product category information per order"},
    {"table_left": "order_items","join_key": "product_id","table_right": "products","join_type": "LEFT","cardinality": "M:1","notes": "Each order item references one product; a product can appear in many order items"},
    {"table_left": "order_items","join_key": "seller_id","table_right": "sellers","join_type": "LEFT","cardinality": "M:1","notes": "Each order item is sold by one seller; a seller can sell many order items"}
])

print("===== RELATIONSHIP MAPPING DOCUMENT =====")
print(relationship_map.to_string(index=False))

relationship_map.to_csv(
    r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\raw_integrated\relationship_mapping.csv",
    index=False
)

===== RELATIONSHIP MAPPING DOCUMENT =====
 table_left                 join_key table_right join_type cardinality                                                                            notes
     orders              customer_id   customers      LEFT         M:1                                               Each order belongs to one customer
     orders                 order_id   items_agg      LEFT         1:1                                          Aggregated order item metrics per order
     orders                 order_id     pay_agg      LEFT         1:1                                             Aggregated payment metrics per order
     orders                 order_id     rev_agg      LEFT         1:1                                              Aggregated review metrics per order
  customers customer_zip_code_prefix     geo_agg      LEFT         M:1                              Customer location attributes aggregated by zip code
     orders                 order_id     cat_a

**Insights:** 

The relationship map captures 8 join paths across all source tables:- 5 are 1:1 at order level after aggregation and 3 are M:1 dimension lookups (customers, products, sellers). 

All joins use a LEFT strategy so the 10,000-order base is retained even when downstream tables have partial coverage. 

The two items-level joins (products and sellers) operate on the 21,838-row order_items grain before the order-level rollup. 

This structure keeps the master table at exactly one row per order without any silent row inflation.